训练时我们关心的是：
* 吞吐
* 反向传播
* 大 batch
* 更新参数

推理时我们关心的是：
* 首 token 延迟
* 每 token 延迟
* KV cache 占用
* 并发请求
* 服务吞吐

这也是为什么现代 serving 框架会大量强调：
* KV cache
* continuous batching
* prefix caching
* chunked prefill
* speculative decoding
* paged attention

比如 vLLM 官方文档把 PagedAttention、continuous batching、chunked prefill、speculative decoding、prefix caching 都列成核心特性；SGLang 官方文档也把 RadixAttention、Prefill-Decode 分离、连续批处理、分页注意力、投机采样和量化列成主打能力。
## 6.1 自回归生成机制
decoder-only 语言模型的生成，本质上是一个 **离散时间序列上的 next-token prediction**。
$$
x_{t+1} \sim P(x \mid x_{1:t})
$$
也就是给当前前缀 (x_{1:t})，模型输出下一个 token 的分布；采一个 token，拼回前缀末尾，再继续下一轮。
```
prompt
  ↓
model(prompt) -> logits at last position
  ↓
sample next token
  ↓
append to sequence
  ↓
repeat
```
训练时，我们可以一整个 [B, S] 一起并行算。

但推理时，哪怕一层 Transformer 的矩阵乘法很并行，token 之间的时间顺序仍然是自回归串行依赖。

这正是后面所有推理优化技术存在的根源：它们几乎都在想办法缓解“必须一个个生成”的串行瓶颈。
> Speculative Decoding 论文就是从这里出发，明确把目标定义成“在不改变输出分布的前提下，加速自回归解码”。
> 很火的 Diffusion Transformer 倒是玩了个文字游戏，实际上更像Bert
现代推理系统几乎一定会引入 **KV cache**，也就是把前面 token 已经算过的 Key/Value 缓存下来，下一步只增量算新 token。TensorRT-LLM 的就把它定义为“存储先前计算好的 key-value 对，以便在生成中复用，避免冗余计算”；并且它还支持跨请求复用、offloading、prioritized eviction，以及对 MQA/GQA 这类注意力优化的支持。

此外，由于模型训练时的最大上下文长度有限，RoPE / attention cache / position 索引通常也都围绕这个窗口设计的，所以如果生成序列越来越长，最直接的处理方式就是 **滑动窗口**，`idx_cond = generated[:, -self.context_length:]` 只保留最后 context_length 个 token 继续喂模型。
## 6.2 采样策略
如果每一步都直接拿最大概率 token，也就是 greedy decoding，模型会非常稳定，但往往也非常无聊，过于保守、重复、甚至机械。

所以推理时我们通常不会直接对 logits 做 argmax，而是先对分布做一些 **整形**，再去采样。
1. Temperature：调分布的尖锐程度
2. Top-k：固定候选池大小
3. Top-p / Nucleus Sampling：固定候选池的累计概率质量

### 6.2.1 Temperature
其实蛮直观的，**低温采样** 的时候熵比较低，自然答案就很确定。
$$
P_i = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}
$$
- logits 是 (z_i)，τ 是 temperature超参
* (\tau > 1)：分布变平，随机性更强
* (\tau < 1)：分布变尖，输出更保守
* (\tau \to 0)：越来越接近 greedy
> Temperature 不改变“谁排第一”的趋势，但会改变“第一名和后面差多少”
> 很多工程实现会在除法里加个很小的 epsilon，避免 temperature 极小或误传 0 时数值出事
### 6.2.2 Top-k
1. 找出 logits 最大的前 `k` 个 token
2. 其余 token 直接设为 `-inf`
3. 再 softmax + 采样

它的优点是实现简单、控制直接。
缺点是它不够自适应，因为不同上下文的确定性差异很大。有的上下文里前 3 个候选已经足够，有的上下文里固定前 50 个又不一定合理。


In [13]:
import torch
def top_k_filter(logits: torch.Tensor, k: int | None) -> torch.Tensor:
    # 1. 边界情况：如果 k 无效，直接返回原 logits
    if k is None or k <= 0 or k >= logits.size(-1):
        return logits
    
    values, _ = torch.topk(logits, k, dim=-1)
    # torch.topk(input, k, dim=-1)：返回 (values, indices)
    # values：前 k 大的 logits 值
    # indices：前 k 大的 logits 对应的索引（这里我们暂时不用 indices）

    # values[..., -1]：取最后一个元素，也就是第 k 大的 logits
    # [..., None]：保持维度不变，方便后面广播比较
    # 例子：logits 形状 [1, 5]，values 形状 [1, 3]，kth 形状 [1, 1]
    # 最终找到第 k 大的 logits 值（门槛值）
    kth = values[..., -1, None]

    # 把小于 kth 的 logits 全部设为 -inf
    results = logits.masked_fill(logits < kth, float("-inf"))
    return results


### 6.2.3 Top-p（Nucleus Sampling）
这是目前文本生成里最常用、效果最好的采样方法之一，比 Top-k 更有「语义弹性」。
不固定保留多少个 token，而是保留「累计概率刚好超过 p」的最小集合：
- 如果模型非常确定（比如前 2 个 token 概率加起来就有 95%），候选集合就很小（只有 2 个）。
- 如果模型不确定（比如前 10 个 token 概率加起来才到 90%），候选集合就自然变大（有 10 个）。

1. 对 logits 按降序排序
2. 计算 softmax 概率并做累积求和
3. 找到累计概率第一次超过 p 的位置
4. 把后面的 token 全部掐掉
5. 把掐掉后的 logits 再 softmax 采样

In [16]:
def top_p_filter(logits: torch.Tensor, p: float) -> torch.Tensor:
    if p >= 1.0:
        return logits

    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    # 原 logits:        [10, 9, 8, 1, 0.5] (索引 0:A, 1:B, 2:C, 3:D, 4:E)
    # sorted_logits:    [10, 9, 8, 1, 0.5]
    # sorted_indices:   [0, 1, 2, 3, 4] (如果原 logits 是乱序的，这里会变)

    sorted_prob = torch.softmax(sorted_logits, dim=-1)  # sorted_probs [0.5, 0.3, 0.1, 0.08, 0.02]
    cumulative_prob = torch.cumsum(sorted_prob, dim=-1)

    # 找出需要移除的 token（在排序后的空间里）
    mask = cumulative_prob > p

    # 把标记往后移一位，保证「刚好超过 p」的那个 token 被保留
    # 把第 0 个位置的值赋给第 1 个，第 1 个赋给第 2 个...（整体右移）
    mask[..., 1:] = mask[..., :-1].clone()
    mask[..., 0] = False        # 永远保留第一个位置（概率最大的token）
    # 右移前：[False, False, False, True, True]
    # 右移后：[False, False, False, False, True] （注意第 3 位变成了 False）

    # 把「排序后的移除标记」填回「原始索引空间」
    indices_to_remove = torch.zeros_like(logits, dtype=torch.bool)

    # 把「排序后要移除的位置」(sorted_indices_to_remove) 映射回「原始索引要移除的位置」(indices_to_remove)
    # scatter_(dim, index, src)：把 src 的值按照 index 的索引填到 dim 维度上
    indices_to_remove.scatter_(dim=-1, index=sorted_indices, src=mask)

    return logits.masked_fill(indices_to_remove, float("-inf"))

In [ ]:
# 假设模型输出的 logits：
logits = torch.tensor([[10.0, 9.0, 8.0, 1.0, 0.5]]) 
# print(f"原 logits: {logits}")

# 用 Top-p，p=0.9
filtered_logits = top_p_filter(logits, p=0.9)
# print(f"Top-p (p=0.9) 后: {filtered_logits}")

原 logits: tensor([[10.0000,  9.0000,  8.0000,  1.0000,  0.5000]])
Top-p (p=0.9) 后: tensor([[10.,  9., -inf, -inf, -inf]])


In [ ]:
def sample_next_token(
    logits: torch.Tensor,
    temperature: float=1.0,
    top_k: int|None=None,
    top_p: float=1.0
) -> torch.Tensor:
    if temperature <= 0:
        # greedy
        return torch.argmax(logits, dim=-1, keepdim=True)
    logits = logits / temperature
    logits = top_k_filter(logits, top_k)
    logits = top_p_filter(logits, top_p)

    probs = torch.softmax(logits, dim=-1)
    # multinomial 是多项式采样：根据 probs 里的概率分布，随机采样 num_samples 个 token
    return torch.multinomial(probs, num_samples=1)

In [18]:
@torch.no_grad()
def generate(
    self,
    prompt_ids: torch.Tensor,       # [Batch_Size, Prompt_Length]
    max_new_tokens: int,            # 最多生成多少个新 token
    eos_token_id: int | None=None,  # 结束符 token ID，遇到就提前停止
    temperature: float=1.0,
    top_k: int | None=None,
    top_p: float = 1.0,
) -> torch.Tensor:
    self.eval()
    # 作用：
    # 1. 关闭 Dropout（训练时随机丢弃神经元，推理时必须全开）
    # 2. LayerNorm/BatchNorm 使用训练时累积的统计量，而不是当前 batch 的统计量

    generated = prompt_ids.clone()

    for _ in range(max_new_tokens):     # 最多循环 max_new_tokens 次
        # 滑动窗口
        idx_cond = generated[:, -self.context_length:]  # 取最后 context_length 个 token
        # 输入 idx_cond = [100, 200, 300]（形状 [1, 3]）
        # 输出 logits 形状 [1, 3, 10000]（假设词表大小 10000）
        logits = self(idx_cond)

        # 取 Batch 维度所有元素，序列维度的最后一个元素，词表维度所有元素
        last_logits = logits[:, -1, :]

        next_token = sample_next_token(
            last_logits,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )

        # 把新 token 拼回上下文
        generated = torch.cat([generated, next_token], dim=-1)
        
        # 如果设置了 eos_token_id
        # 而且 Batch 里所有句子都生成了结束符，就提前停止循环，不用生成满 max_new_tokens 个 token
        # .all()：Batch 里所有句子都满足条件才停止
        if eos_token_id is not None and (next_token == eos_token_id).all():
            break

    return generated

| 轮次 | generated（当前上下文） | idx_cond（滑动窗口） | next_token（采样结果） | 操作 |
|------|-------------------------|----------------------|------------------------|------|
| 初始 | [100, 200, 300] | - | - | 输入提示词"我爱吃" |
| 1 | [100, 200, 300] | [100, 200, 300] | 400（包） | 拼回，generated变成[100,200,300,400] |
| 2 | [100, 200, 300, 400] | [100, 200, 300, 400] | 500（子） | 拼回，generated变成[100,200,300,400,500] |
| 3 | [100, 200, 300, 400, 500] | [100, 200, 300, 400, 500] | 600（很） | 拼回，generated变成[100,200,300,400,500,600] |
| 4 | [100, 200, 300, 400, 500, 600] | [200, 300, 400, 500, 600]（滑动窗口取最后5个） | 700（香） | 拼回，generated变成[...700]，循环结束 |
| 最终 | [100, 200, 300, 400, 500, 600, 700] | - | - | 返回完整token IDs |

## 6.3 KV Cache
前面我们提到，大模型是「自回归生成」的：你输入「我爱吃」，它先预测下一个token是「苹」，再把「我爱吃苹」作为输入，预测下一个是「果」，以此类推。

在自注意力层里，生成第N个token时，需要用到前面N-1个token的 **K（键向量）和V（值向量）**。如果每次生成都重新算一遍前面所有的K和V，计算量会指数级增长，速度慢到爆炸。

然而，为什么每一步都把整段前缀重新算一遍？前面那些 token 的 K/V 不是已经算过了吗？

这就是 KV Cache 的起点。

本质上是把生成拆成两段：
- 第一段：**Prefill**
    - 整段 prompt 跑一次，得到各层历史 token 的 K/V。
- 第二段：**Decode**
    - 后面每一步只喂 新 token，同时把新的 K/V 追加到 cache 里。

## 6.4 PagedAttention
然而一旦你开始服务很多并发请求，KV cache 会迅速变成 显存碎片和动态分配管理问题。

假定大模型推理服务 = 外卖餐馆后厨。
- 1个用户请求 = 1个外卖订单
- KV Cache = 订单的备菜（每个token对应一份备菜，生成新token时必须复用之前的备菜）
- 显存 = 后厨的案板空间

一个订单来了，**必须给它分配一块连续的大案板**：
- 比如用户输入了256个token的prompt，最多要生成1024个token，传统做法会直接给它分配一块「能放下1024份备菜」的连续案板空间。
- 整个订单的KV Cache，必须存在这块连续的空间里，不能拆分。

### 高并发问题
1. **严重的显存碎片**
订单是动态来、动态结束的，连续空间的分配释放会快速产生大量无法利用的碎片：
- 订单A要1024份备菜，占了一块1024位的连续案板，用完释放了。
- 这时来了订单B，要768份备菜，用了这块空间里的768位，剩下256位的小空间，没有任何订单能用上（几乎没有订单刚好只需要256位）。
- 多来几次，案板上全是这种「用不了、扔不掉」的小碎片——**明明总案板空间够，但没有连续的大空间，新订单根本分配不了，直接OOM炸显存**。

2. **极大的空间浪费**
用户的生成长度是完全不确定的：
- 你提前给订单分配了1024位的连续空间，但用户可能只生成了200个token就结束了对话。
- 剩下的824位空间，从订单开始到结束，一直被占着，完全无法给其他订单用，**显存利用率直接跌到20%以下**。

3. **并发上不去**
因为碎片和浪费，你的显存明明能同时跑100个订单，结果只能接20个，吞吐直接被锁死。这就是为什么原生Hugging Face Transformers的推理服务，并发能力极差的核心原因。

### 操作系统分页思想
操作系统早就解决了「内存碎片、连续空间分配」的问题——就是**虚拟内存分页机制**：
- 把物理内存切成固定大小的「页（Page）」（比如4KB一页）。
- 程序需要内存时，不用分配连续的物理页，给它分配一堆离散的页就行。
- 用一张「页表」记录「程序的逻辑地址」和「物理页位置」的映射关系。
- 这样几乎没有内存碎片，内存利用率直接拉满。

PagedAttention把这个思路1:1搬到了KV Cache的管理上，核心设计只有3点：

1. **把显存切成固定大小的「Block（块）」**
把整个GPU显存，切成**固定大小的KV Cache Block**，每个Block刚好能存固定数量token的KV Cache（vLLM默认是16个token一个Block）。
- 类比：把后厨的大案板，全切成**固定大小的小碟子**，每个碟子刚好放1份备菜。

2. **离散分配，不用连续空间**
一个请求来了，不管它要生成多少token，都不用给它分配连续的大空间，只需要按需分配一个个离散的Block就行。
- 比如一个请求要生成200个token，只需要给它分配13个Block（13×16=208，刚好装下），这些Block可以在显存的任意位置，不用连续。
- 类比：一个订单要200份备菜，不用给它划一块连续的大案板，只需要给它200个小碟子，碟子可以散落在后厨的任意位置。

3. **用「页表」管理映射关系**
给每个请求维护一张「页表」，记录「这个请求的第N个token的KV Cache，存在哪个Block的哪个位置」。
- 计算注意力时，直接通过页表找到对应Block里的KV数据，拼接起来计算就行。
- 类比：给每个订单一张「菜单对照表」，记录这个订单的第1道菜在哪个碟子，第2道菜在哪个碟子，厨师做菜时直接按表找碟子就行。

### 收益
1. **Near-Zero Waste（近乎零浪费）**
- 传统KV Cache：显存利用率通常只有20%~30%，大部分空间被浪费和碎片吃掉。
- PagedAttention：显存利用率能做到90%以上，只有订单最后一个Block可能有几个空位的微小浪费，几乎可以忽略。
- 直观效果：同样一张24G的RTX 4090，传统方法只能同时跑4个7B模型的并发请求，用PagedAttention能同时跑30+个。

2. **天然支持Prompt共享（Prefix Caching）**
这是PagedAttention带来的 **「意外之喜」**，也是现在业务场景里收益最大的特性：
- 业务场景里，绝大多数请求都有**相同的前缀**：比如统一的系统prompt（「你是一个乐于助人的AI助手」）、RAG里检索到的同一篇文档、多轮对话里的历史内容。
- 传统KV Cache：每个请求都要单独存一遍这些相同前缀的KV Cache，100个请求就存100遍，显存直接浪费100倍。
- PagedAttention：这些相同前缀的KV Cache，只需要计算和存储一次，放在几个固定的Block里，所有请求的页表都直接映射到这几个Block上，**100个请求共享同一份KV Cache**。
- 直观效果：在RAG、客服机器人等场景，系统prompt+文档动辄几千个token，共享后显存占用直接砍半，并发再翻一倍。

3. **吞吐爆炸式提升，同时不损失模型精度**
- PagedAttention完全没有改变注意力的计算逻辑，只是优化了KV Cache的存储和读取方式，**对模型生成精度零影响**。
- 因为显存利用率拉满，能同时处理的请求数量翻了10~20倍，再配合 **连续批处理（Continuous Batching）**，vLLM的吞吐能比原生Hugging Face高几十倍。
- 这就是为什么现在所有工业级推理引擎（vLLM、SGLang、TGI、TensorRT-LLM），全都把PagedAttention作为核心标配。

## 6.5 Speculative Decoding（投机解码/推测解码）
自回归生成一次只能产 1 个 token，太串行了。哪怕你的GPU有上万个核心，算力极强，算1个token只用到了1%~5%的算力，剩下95%的算力都在闲置——因为串行逻辑锁死了，你必须等前一个token出来，才能算下一个。能不能先猜多个，再让大模型一起验？

这是 **speculative decoding** 论文的结论，就是 **用一个低成本的小模型先「猜一串候选token」，再让大模型一次性并行验证这串token**，把原本需要串行K次的大模型前向计算，压缩成1次并行计算，在 **完全不改变大模型输出分布、零精度损失** 的前提下，实现2~3倍的推理加速。

具体来说，前置需要两个 **同词表、同tokenizer** 的模型：
1.  **目标模型 T**：你原本要加速的大模型（比如70B大模型），生成质量高、单步推理慢。
2.  **草稿模型 D**：比T小很多的同架构模型，生成速度是T的5~10倍，和T的输出分布越接近越好（接受率越高，加速效果越好）。

### 步骤1：草稿模型快速生成K个候选token
用草稿模型D，基于当前上下文，**串行生成K个连续的候选token**（行业常用K=4~8），同时把生成过程中的KV Cache缓存好。
- 例子：当前上下文是「我今天早上吃了」，D快速生成4个候选：`["包子", "喝了", "豆浆", "非常"]`。注意这里是「接下来要生成的一整段连续文本」！
- 核心：D的速度极快，生成K个token的时间，远小于T生成1个token的时间，几乎不增加额外耗时。

### 步骤2：大模型一次性并行计算全量logits
把**「原始上下文 + K个候选token」完整拼接成一个序列**，一次性丢给目标模型T。
- 例子：输入变成「我今天早上吃了包子喝了豆浆非常」，T只需要做**1次前向传播**，就能并行算出这个完整序列里，所有位置的logits（包括4个候选token对应的位置）。

### 步骤3：逐token验证，严格的接受-拒绝逻辑
从第一个候选token开始，逐个对比「草稿模型猜的token」和「目标模型的输出分布」，执行严格的 **接受-拒绝采样**：
1.  从第1个候选token开始，检查D猜的token，是否符合T的输出分布。
2.  一直找到**第一个T不接受的token**：
    - 前面所有验证通过的token，全部保留输出。
    - 第一个不接受的位置，用T自己采样的token替换。
    - 这个位置之后的所有候选token，全部扔掉。
3.  如果K个候选全被接受，就额外多输出1个T自己采样的token，保证流程推进。

- 候选序列：`["包子", "喝了", "豆浆", "非常"]`
- T验证结果：前3个全接受，第4个「非常」不接受，T认为应该是「很」。
- 最终输出：`["包子", "喝了", "豆浆", "很"]`，前3个直接用，第4个用T的结果，后面的候选全部丢弃。

### 步骤4：KV Cache复用，更新上下文
把验证通过的token，更新到上下文里，同时复用草稿模型和目标模型已经算好的KV Cache，不用从头重算，进一步节省时间。

### 步骤5：循环重复，直到生成结束
用新的上下文，回到步骤1，让草稿模型继续猜下一批候选token，直到生成终止符<eos>。

草稿模型只负责「猜候选」，最终要不要用这个token，100%由目标模型的分布决定。哪怕草稿模型猜的token是对的，目标模型也会严格按自己的概率分布判断要不要接受，不会因为草稿猜了就直接用。最终的生成结果，完全等价于你直接用目标模型纯自回归生成的结果，**没有任何精度损失，也不会改变生成的随机性**。量化、蒸馏等加速方法或多或少会有精度损失，而投机解码是 **无损加速**。

### 边界限制
1.  **加速比有上限**：理论最大加速比等于候选数K，实际因为接受率不可能100%，通常稳定在2~3倍，很难超过5倍。
2.  **依赖草稿模型的匹配度**：草稿模型和目标模型的分布越接近，token接受率越高，加速效果越好；如果草稿猜的全错，反而会因为额外的草稿前向变慢。
3.  **需要额外显存**：需要同时加载目标模型和草稿模型，比如跑70B的目标模型，还要额外加载一个8B的草稿模型，对显存有更高要求（可以通过量化解决）。

> 另外 vllm 有2个值得说道的工程优化
> - **树状投机解码**：不再只猜一条线性的候选序列，而是让草稿模型生成一个「候选token树」，一次给大模型提交几十上百个候选路径，大模型一次验证就能接受更多token，把加速比从2~3倍提升到5倍以上。
> - **无额外模型的自投机解码**：不用单独部署一个小的草稿模型，直接用大模型的前几层做草稿，后几层做目标，两个模型共享权重，不用额外的显存开销，部署门槛大幅降低。

## 6.6 StreamingLLM
你大模型训练时有一个固定的上下文窗口（比如Llama2是4096token），推理时一旦输入超过这个窗口，就会面临两个选择：
1.  **全量重算**：把超长文本全丢进去，KV Cache显存直接爆炸，速度慢到无法用；
2.  **朴素滑动窗口**：模型训练的上下文窗口是4096token，意味着它这辈子只见过最多4096个token的连续输入。当你要处理10000个token的超长流式输入时，朴素滑动窗口的做法是：永远只保留 **最近的4096个token** 的KV Cache，前面的旧token直接删掉。比如输入到第4097个token时，把第0个token的KV Cache删掉，只留1~4097的，窗口永远固定4096大小。


论文里做了大量实验，发现一个致命现象：**只要你删掉输入最开头的几个token，模型的注意力计算会直接乱掉，生成质量瞬间崩盘**。

哪怕你删掉的是和当前内容完全无关的、几千个token之前的起始符`<bos>`，模型也会直接崩。

你看一本1000页的长篇小说，看到第900页的时候，哪怕当前页的内容和书名、扉页完全无关，你的眼睛也会不自觉地瞟一眼书的封面/书名——你必须看到这个「锚点」，才能稳定地知道「我正在看哪本书、我在看什么」，不然就会走神、完全看不懂当前的内容。

Transformer模型里，**输入最开头的几个token（尤其是`<bos>`起始符），就是这个必须存在的「锚点」**，论文把它叫做 **Attention Sink（注意力水槽）**。

论文通过注意力热力图分析，发现了一个通用规律：
> 不管输入的文本是什么、当前生成到哪个位置，**模型的所有注意力头，都会把大量的冗余注意力权重，分配给输入最开头的几个token** —— 哪怕这些token和当前生成的内容没有任何语义关系。

这和Transformer注意力的Softmax机制强相关：
1.  注意力计算的最后一步是Softmax，它的核心规则是：**所有位置的注意力权重加起来必须等于1**。
2.  模型在训练时，对于和当前token无关的位置，本应该让它们的注意力权重趋近于0。但Softmax的特性是：哪怕你给无关位置的logits设为负无穷，Softmax也会把权重平均分配，无法真正归零。
3.  于是模型学会了一个「偷懒但高效的技巧」：**把所有无关位置的冗余注意力权重，全部丢给最开头的几个token**。
    - 这些开头的token就像一个「水槽」，所有没用的垃圾权重都流进去，不会乱流到有效内容的位置上。
    - 这样一来，有效内容的注意力权重就干净、稳定了，模型的生成效果才有保证。
    - 水槽没了，垃圾权重就会乱流到所有有效内容的位置上，注意力计算就会乱套。

**永远保留Attention Sink**，模型有固定的水槽倒冗余权重，注意力计算完全稳定，和训练时的行为100%一致，哪怕输入几百万token，困惑度也不会爆炸，生成质量完全正常。

| 输入长度 | KV Cache组成 | 窗口大小 |
|----------|--------------|----------|
| 1000token | 前4个Sink Token + 最近996个token | 4096 |
| 5000token | 前4个Sink Token + 最近4092个token（删掉了第5~908个旧token） | 4096 |
| 400万token | 前4个Sink Token + 最近4092个token（只删滑动窗口里的旧token） | 4096 |

BTW，StreamingLLM能让模型 **在超长流式输入下稳定生成不崩盘**，但 **不能让模型理解超过滑动窗口的历史内容**。

还有一些全量长上下文的手段：
| 特性 | StreamingLLM | RoPE扩展/NTK缩放/LongLoRA |
|------|--------------|-----------------------------|
| 核心能力 | 超长流式输入下稳定生成 | 全量超长上下文的理解和回顾 |
| 是否需要微调 | 完全不用，零成本 | 大多需要微调/位置编码修改 |
| KV Cache显存 | 固定不变（等于训练窗口大小） | 随上下文长度线性增长 |
| 最佳场景 | 无限流式对话、实时文本生成 | 超长文档总结、全量内容检索 |

## 6.7 Quantized Inference
量化的本质，是 **将高精度浮点数映射到低比特离散数值集，在可控精度损失下，压缩显存/硬盘占用、利用硬件原生低比特指令加速推理**。

### 存储格式
所有量化的前提，是理解不同数值格式的底层结构。
一个浮点数由三部分组成：
`数值 = (-1)^符号位S × (1 + 尾数位M/2^尾数位宽) × 2^(指数位E - 偏移量)`
- **符号位S**：1位，0为正，1为负
- **指数位E**：决定数值的**动态范围**（能表示的最大值/最小值）
- **尾数位M**：决定数值的**精度**（小数点后能保留的有效数字）

| 精度格式 | 位宽结构 | 单参数占用 | 动态范围 | 核心精度特性 | 硬件支持 | 7B模型体积 | 核心适用场景 |
|----------|----------|------------|----------|--------------|----------|-------------|--------------|
| FP32（单精度浮点数） | 1S+8E+23M | 4字节 | ~1e-38 ~ 1e38 | 精度最高，无溢出风险 | 全平台通用 | ~28GB | 模型训练、高精度校验、算子开发 |
| FP16（IEEE半精度） | 1S+5E+10M | 2字节 | ~6e-5 ~ 6e4 | 尾数精度高，但动态范围极小，极易梯度下溢/溢出 | 全系列NVIDIA GPU、AMD GPU | ~14GB | 老GPU推理、不支持BF16的硬件，训练需配合梯度缩放（AMP GradScaler） |
| BF16（BFloat16） | 1S+8E+7M | 2字节 | ~1e-38 ~ 1e38 | 指数位与FP32完全一致，动态范围和FP32相同，无溢出风险；尾数精度略低于FP16，但大模型冗余度可覆盖 | NVIDIA A100+/AMD MI200+/苹果M系列+ | ~14GB | 大模型训练/微调的默认格式、高端GPU推理，无需梯度缩放，训练稳定性远超FP16 |
| FP8（E4M3） | 1S+4E+3M | 1字节 | ~4e-2 ~ 448 | 平衡动态范围与精度，硬件原生加速 | NVIDIA H100+/AMD MI300+ | ~7GB | 工业界线上服务首选，权重量化+KV Cache量化通用，几乎无精度损失 |
| FP8（E5M2） | 1S+5E+2M | 1字节 | ~6e-5 ~ 57344 | 动态范围更大，精度略低 | 同上 | ~7GB | 动态范围要求高的场景，如KV Cache量化 |
| INT8（8位整数） | 8位整数 | 1字节 | -128 ~ 127（对称） | 需缩放因子映射浮点数，压缩比高，硬件加速成熟 | 全平台支持Tensor Core的GPU/CPU | ~7GB | 全量化线上服务、低显存设备推理 |
| INT4（4位整数） | 4位整数 | 0.5字节 | -8 ~ 7（对称） | 极致压缩，需细粒度分组量化保证精度 | 主流GPU/llama.cpp全支持 | ~3.5GB | 本地部署、消费级显卡跑大模型的黄金标准 |
| INT2（2位整数） | 2位整数 | 0.25字节 | -2 ~ 1（对称） | 极限压缩，精度损失较大 | llama.cpp/小众框架支持 | ~1.8GB | 极低显存设备（手机/老电脑）、极限压缩场景 |

- FP16的指数位只有5位，最大值仅65536，最小值6e-5，训练时梯度很容易下溢变成0，或激活溢出变成inf，必须用GradScaler做梯度缩放。
- BF16的指数位和FP32一样是8位，动态范围和FP32完全一致，绝对不会出现溢出/下溢，训练时无需梯度缩放，稳定性碾压FP16。

### 分类
#### 1. 按量化对象分类
| 分类 | 量化目标 | 核心价值 |
|------|----------|----------|
| 权重量化 | 模型固定的权重参数（训练后不再变化） | 压缩模型本体体积，降低计算量，让模型塞进小显存设备 |
| KV Cache量化 | 推理过程中动态生成的K/V向量 | 降低长上下文推理的显存占用，提升并发吞吐，解决长文本显存爆炸问题 |
| 激活量化 | 模型推理时每层输出的激活值 | 实现端到端全低比特计算，最大化硬件加速，提升线上服务吞吐 |

#### 2. 按量化时机分类
- **PTQ（Post-Training Quantization，后训练量化）**：模型训练完成后，用少量校准数据完成量化，无需微调，门槛极低，是目前最主流的方案（代表：GPTQ、AWQ、SmoothQuant）。
  - GPTQ：让量化后的权重输出，与原始FP16权重输出的均方误差（MSE）最小
  - AWQ：保护好这1%的「显著权重」，剩下99%的权重可以大胆量化，精度几乎不会下降
  - SmoothQuant：上两者仅解决了权重量化，但大模型的激活分布极不均匀，只有实现「权重+激活全INT8量化」，才能利用Tensor Core实现端到端的硬件加速，吞吐翻倍
- **QAT（Quantization-Aware Training，量化感知训练）**：训练/微调时就模拟量化噪声，让模型适应低比特量化，精度更高，但训练成本极高，仅用于极致压缩场景。

#### 3. 按量化规则分类
- **对称量化**：零点固定为0，仅用缩放因子映射浮点数与整数，计算速度快，无额外偏移开销，适合权重量化。
- **非对称量化**：包含缩放因子+零点，能更好地贴合数据分布，精度更高，计算开销略大，适合激活量化。

#### 4. 按量化粒度分类
粒度越细，精度越高，计算开销越大：
`逐张量 < 逐通道 < 逐组（Group） < 逐token`
- 逐张量：整个权重矩阵用一套缩放因子，计算最快，精度最低。
- 逐组（Group）：每N个权重用一套缩放因子（如GPTQ常用group_size=128），是精度与速度的黄金平衡，目前所有主流4bit量化均采用此方案。


### KV Cache量化
`显存占用 = batch_size × 上下文长度 × 层数 × 隐藏层维度 × 2（K和V） × 单数据字节数`

一个真实的长上下文场景（7B模型，32层，4096维度）：
- 普通短对话：上下文长度4096，batch size=1，FP16格式 → KV Cache仅占 ~200MB
- 长文档推理：上下文长度128K（比如读一整本书），batch size=1，FP16格式 → KV Cache直接占到 **~64GB**！

**长上下文推理里，先炸的往往不是权重，而是KV cache**。

因此需要把推理过程中动态生成的K/V向量，从FP16量化成FP8/INT4，显存占用直接砍半甚至砍到1/4。
- 刚才128K上下文的64GB KV Cache，量化成FP8后直接变成32GB，再配合其他优化，就能在普通消费级显卡上跑长文档推理了。
- 对于推理服务（比如用vLLM/SGLang搭API），KV Cache量化后，同样的显存能同时接更多的请求，吞吐直接翻倍。

## 6.8 推理图景
# 主流大模型推理框架 全维度对比总表
| 框架名称 | 官方核心定位 | 核心能力矩阵 | 一句话精准定位 | 最佳适用场景 | 官方参考链接 |
|----------|--------------|--------------|----------------|--------------|--------------|
| **vLLM** | 简单、高速、低成本的全模态大模型推理与服务库，是开源社区通用高吞吐服务的事实标准底座，核心依托PagedAttention技术实现极致显存利用率与吞吐性能 | 1. 核心创新：PagedAttention分页注意力，彻底解决KV Cache显存碎片问题，显存利用率从20%提升至90%以上<br>2. 调度优化：连续批处理（Continuous Batching）、分块预填充（Chunked Prefill）、前缀缓存（Prefix Caching）<br>3. 高级特性：多方案投机解码、全系列量化支持（GPTQ/AWQ/SmoothQuant/FP8）、多LoRA服务、分布式推理<br>4. 生态兼容：原生OpenAI兼容API、支持100+主流模型架构、全模态推理（文本/图像/音频/视频，vLLM-Omni）<br>5. 跨硬件支持：NVIDIA/AMD/TPU/Intel/华为昇腾等10+硬件平台 | 快速搭建高吞吐、高并发的通用推理服务，不想从零自研runtime，是目前工业界通用场景最自然的第一选择 | 通用线上高并发推理服务、多模态模型部署、初创项目快速上线、异构硬件环境适配、长上下文推理场景 | [vLLM 官方文档](https://docs.vllm.com.cn/) |
| **TensorRT-LLM** | NVIDIA官方开源、专为NVIDIA GPU深度软硬协同优化的大语言模型推理库，目标是在NVIDIA硬件上实现推理性能的天花板 | 1. 核心优势：基于TensorRT引擎的硬件感知内核自动调优、算子融合，H100上可实现最高8倍推理加速<br>2. KV管理：高效分页KV缓存、智能块复用、显存/主机Offload、优先级驱逐策略<br>3. 并行能力：多GPU多节点张量并行、流水线并行、专家并行，无缝支持千亿级大模型分布式推理<br>4. 特性支持：In-Flight Batching、MQA/GQA/MLA注意力、全系列量化（FP8/NVFP4/INT4 AWQ/INT8 SmoothQuant）、多方案投机解码（EAGLE-3等）、LoRA多适配器支持 | 部署环境为纯NVIDIA技术栈，且需要把延迟、吞吐、KV管理都压到极致，是NVIDIA硬件上的性能天花板首选 | 纯NVIDIA GPU环境的企业级线上服务、H100/B100等高端卡极致性能压榨、超大规模高并发低延迟推理服务、FP8等低比特极致量化场景 | [TensorRT-LLM 官方文档](https://developer.nvidia.cn/tensorrt-llm) |
| **SGLang** | 面向结构化LLM程序的高性能推理部署框架，以RadixAttention为核心创新，激进落地各类前沿推理系统优化技巧，实现极致的缓存复用与结构化生成性能 | 1. 核心创新：RadixAttention基数树KV缓存，跨请求自动复用公共前缀，多轮对话场景缓存命中率提升3-5倍<br>2. 调度优化：Prefill-Decode分离、连续批处理、分页注意力、分块预填充、缓存感知调度<br>3. 高级特性：投机采样、全量化支持与量化KV缓存、多LoRA服务、压缩有限状态机加速结构化输出、API推测执行<br>4. 生态兼容：原生OpenAI兼容API、支持主流开源大模型、多模态推理能力 | 相比vLLM的强通用底座，SGLang更偏向将推理系统的各类高级技巧产品化，尤其适合前缀缓存、结构化输出、多轮对话、多LoRA服务等场景 | 多轮对话机器人、RAG检索问答、少样本推理、结构化生成（JSON/格式约束）、高重复前缀的高并发服务、思维树等复杂推理场景 | [SGLang 官方文档](https://sglang.readthedocs.io/) |
| **llama.cpp** | 极简、可移植、无依赖的C/C++开源大模型推理框架，以CPU优先的设计哲学，专注于在本地消费级硬件、边缘设备上实现高效的大模型推理 | 1. 格式标准：自研GGUF通用模型格式，单文件打包模型结构、tokenizer、量化权重与元数据，即开即用<br>2. 量化能力：支持从FP16到2bit的全系列低比特量化，针对本地硬件做了精度与速度的极致平衡，7B模型可压缩至1.8GB<br>3. 硬件优化：全平台跨架构支持，针对x86 AVX2、ARM NEON、苹果硅AMX、NVIDIA CUDA做了汇编级指令集优化<br>4. 高级特性：8bit KV Cache量化、投机解码、PagedAttention、StreamingLLM流式长文本支持、原生OpenAI兼容API服务、零依赖单文件部署 | 本地部署、CPU/单机轻量GPU场景、对GGUF格式与低比特量化敏感的本地推理生态首选 | 本地离线部署、消费级笔记本/台式机无高端GPU场景、边缘设备推理、隐私敏感的本地AI应用、跨平台轻量部署 | [llama.cpp 官方GitHub](https://github.com/ggml-org/llama.cpp) |
| **DeepSpeed-Inference** | 微软DeepSpeed生态中专为大模型推理优化的技术集合，核心解决超大规模模型的部署难题，与DeepSpeed训练生态无缝衔接，让原本跑不动的百亿/千亿参数模型在有限硬件上可运行 | 1. 核心能力：ZeRO-Inference显存分片技术，将模型权重分布式存储在多GPU上，突破单卡显存限制，支持100B+超大模型推理<br>2. 性能优化：推理定制化高性能内核、算子融合、张量/流水线并行、MoE稀疏模型专家并行、MoQ量化方案<br>3. 显存扩展：支持CPU/NVMe Offload，进一步突破GPU显存墙，在有限硬件上运行极限规模模型<br>4. 生态衔接：与DeepSpeed训练流程无缝兼容，支持HuggingFace、Megatron训练的模型零修改部署 | 本身就在DeepSpeed大规模训练生态中，或极度关注张量并行、内核注入、训练与部署生态衔接的场景首选 | 100B+超大规模模型推理、与DeepSpeed训练流程无缝衔接的部署、有限GPU上运行超大模型、MoE稀疏模型部署、大规模分布式推理服务 | [DeepSpeed-Inference 官方文档](https://deepspeed.org.cn/inference/) |
